<a href="https://colab.research.google.com/github/ebbettin/UCH_SRL/blob/main/Filter_Full_Length_sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.2 MB/s eta 0:00:00


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
import os
import shutil
from google.colab import files
import pandas as pd
from collections import defaultdict

In [ ]:
# 2. Directories
INPUT_DIR = "/content/input"
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_FILE = os.path.join(OUTPUT_DIR, "processing_summary.txt")
EXCEL_FILE = os.path.join(OUTPUT_DIR, "sequence_classification.xlsx")

# 3. Stop codons
STOP_CODONS = {"TAA", "TAG", "TGA"}

# Track global classification per header
header_classification = defaultdict(set)

# 4. Function to classify and translate sequences
def classify_and_translate(fasta_file):
    full_nuc = []
    full_aa = []
    deg_nuc = []
    deg_aa = []

    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = str(record.seq).upper()

        # Length divisible by 3?
        if len(seq) % 3 != 0:
            deg_nuc.append(record)
            aa_record = record[:]
            aa_record.seq = Seq(seq).translate(to_stop=False)
            deg_aa.append(aa_record)
            continue

        codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
        stop_positions = [i for i, c in enumerate(codons) if c in STOP_CODONS]

        # Translate protein
        aa_record = record[:]
        aa_record.seq = Seq(seq).translate(to_stop=False)

        # Full-length: exactly one stop codon at last codon
        if len(stop_positions) == 1 and stop_positions[0] == len(codons) - 1:
            full_nuc.append(record)
            full_aa.append(aa_record)
        else:
            deg_nuc.append(record)
            deg_aa.append(aa_record)

    return full_nuc, full_aa, deg_nuc, deg_aa


# 5. Process files: write FASTAs, summary, prepare Excel
with open(SUMMARY_FILE, "w") as summary_writer, pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as excel_writer:
    summary_writer.write("FASTA processing summary\n")
    summary_writer.write("="*50 + "\n\n")

    for fname in os.listdir(INPUT_DIR):
        if not fname.lower().endswith((".fa", ".fasta", ".fna")):
            continue

        input_path = os.path.join(INPUT_DIR, fname)
        base = os.path.splitext(fname)[0]

        # Classify sequences
        full_nuc, full_aa, deg_nuc, deg_aa = classify_and_translate(input_path)

        # Write nucleotide/protein FASTAs
        SeqIO.write(full_nuc, os.path.join(OUTPUT_DIR, f"{base}_FullLength_nuc.fasta"), "fasta")
        SeqIO.write(full_aa,  os.path.join(OUTPUT_DIR, f"{base}_FullLength_aa.fasta"), "fasta")
        SeqIO.write(deg_nuc,  os.path.join(OUTPUT_DIR, f"{base}_Degenerated_nuc.fasta"), "fasta")
        SeqIO.write(deg_aa,   os.path.join(OUTPUT_DIR, f"{base}_Degenerated_aa.fasta"), "fasta")

        # Prepare DataFrame for Excel tab
        rows = []
        for record in full_nuc:
            rows.append({"sequence_id": record.id, "group": "FullLength"})
            header_classification[record.id].add("FullLength")
        for record in deg_nuc:
            rows.append({"sequence_id": record.id, "group": "Degenerated"})
            header_classification[record.id].add("Degenerated")

        df = pd.DataFrame(rows)
        tab_name = base[:31]  # Excel sheet name max 31 chars
        df.to_excel(excel_writer, sheet_name=tab_name, index=False)

        # Write console + text summary
        print(f"{fname}: FullLength={len(full_nuc)}, Degenerated={len(deg_nuc)}")
        summary_writer.write(f"{fname}\n")
        summary_writer.write(f"  Full-length sequences: {len(full_nuc)}\n")
        summary_writer.write(f"  Degenerated sequences: {len(deg_nuc)}\n")
        summary_writer.write("-"*40 + "\n")

    # 6. Add Excel tab for headers always FullLength
    always_full = [header for header, groups in header_classification.items() if groups == {"FullLength"}]
    df_summary = pd.DataFrame({"always_FullLength": always_full})
    df_summary.to_excel(excel_writer, sheet_name="Always_FullLength", index=False)

    # Also append to the text summary file
    summary_writer.write("\nHeaders always classified as FullLength:\n")
    summary_writer.write("="*50 + "\n")
    for header in always_full:
        summary_writer.write(f"{header}\n")

TP0346_TPA_TEN_TPE_combined.fasta: FullLength=1788, Degenerated=1
TP0733_TPA_TEN_TPE_combined.fasta: FullLength=1801, Degenerated=0
TP0479_TPA_TEN_TPE_combined.fasta: FullLength=1788, Degenerated=1
TP0619_TPA_TEN_TPE_combined.fasta: FullLength=1798, Degenerated=1
TP0697_TPA_TEN_TPE_combined.fasta: FullLength=1646, Degenerated=155
TP0347_TPA_TEN_TPE_combined.fasta: FullLength=1271, Degenerated=518
TP0126_TPA_TEN_TPE_combined.fasta: FullLength=1789, Degenerated=0
TP0698_TPA_TEN_TPE_combined.fasta: FullLength=1798, Degenerated=3


In [ ]:
import shutil
import os
from google.colab import files

ZIP_PATH = "/content/output_FASTA_results"
shutil.make_archive(ZIP_PATH, "zip", OUTPUT_DIR)
files.download(f"{ZIP_PATH}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>